In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm 
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, accuracy_score, classification_report
from arabert.preprocess import ArabertPreprocessor

# Fixer la graine aléatoire pour la reproductibilité
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")


In [ ]:
# 2. Nom du modèle
Arabert = "aubmindlab/bert-base-arabertv02-twitter"
# 3. Téléchargement et chargement automatique du Tokenizer et du Modèle
print("Téléchargement / chargement du modèle depuis Hugging Face...")
Arabert_tokenizer = AutoTokenizer.from_pretrained(Arabert)
model_arabert = AutoModel.from_pretrained(Arabert)
# 4. Vérification de la dimension
hidden_dim = model_arabert.config.hidden_size
print(f"Modèle chargé avec succès !")
print(f"Dimension cachée (hidden_size) = {hidden_dim}")  # Affiche 768


In [ ]:
from transformers import AutoTokenizer, AutoModel
import transformers.modeling_utils as mu

# 1. Contourner la vérification de version de torch.load pour MARBERT
mu.check_torch_load_is_safe = lambda: None

# 2. Nom du modèle
marbert_name = "UBC-NLP/MARBERTv2"

# 3. Chargement du Tokenizer et du Modèle
print("Chargement de MARBERTv2 en cours...")
model_marbert = AutoModel.from_pretrained(marbert_name)

# 4. Vérification de la dimension
hidden_dim = model_marbert.config.hidden_size
print(f"Modèle chargé avec succès !")
print(f"Dimension cachée (hidden_size) = {hidden_dim}")  # Affiche 768


In [ ]:
# Loading raw data
train_df = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_train.csv")
val_df = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_val.csv")
test_df = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_test.csv")

print(f"Raw splits loaded: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

In [ ]:
# 1. Labels de Sarcasme (booléen -> entier 0 ou 1)
y_train_sarc = train_df['sarcasm'].astype(int).values
y_val_sarc = val_df['sarcasm'].astype(int).values
y_test_sarc = test_df['sarcasm'].astype(int).values
print("Distribution Sarcasme dans Train :")
print(pd.Series(y_train_sarc).value_counts(normalize=True))

In [ ]:
# Loading raw data
train_df = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_train.csv")
val_df = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_val.csv")
test_df = pd.read_csv(r"..\\Notebooks\\data\\aravec_splits\\aravec_test.csv")

print(f"Raw splits loaded: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")

In [ ]:
# 1. Labels de Sarcasme (booléen -> entier 0 ou 1)
y_train_sarc = train_df['sarcasm'].astype(int).values
y_val_sarc = val_df['sarcasm'].astype(int).values
y_test_sarc = test_df['sarcasm'].astype(int).values
print("Distribution Sarcasme dans Train :")
print(pd.Series(y_train_sarc).value_counts(normalize=True))

1. Le problème : Le déséquilibre des classes
Dans votre jeu de données ArSarcasm-v2 :

~83 % des tweets sont non-sarcastiques (classe 0).
~17 % des tweets sont sarcastiques (classe 1).
Avec une fonction de perte standard (CrossEntropyLoss), le modèle gagne "facilement" en prédisant presque toujours la classe 0 : il obtiendra 83 % d'accuracy sans avoir rien appris sur le sarcasme, et son score F1-sarcasme sera proche de 0.

2. La solution : Pénaliser plus lourdement les erreurs sur la classe minoritaire
La perte pondérée consiste à donner un poids multiplicateur $w_1 > 1$ aux erreurs commises sur la classe sarcastique :

Si le modèle se trompe sur un tweet non-sarcastique $\rightarrow$ pénalité normale ($w_0 = 1.0$).
Si le modèle rate un tweet sarcastique $\rightarrow$ pénalité multipliée par $w_1$ (ex: $w_1 \approx 5.0$).

In [ ]:
# Calcul du poids pour la perte pondérée de E1
num_neg = (y_train_sarc == 0).sum()
num_pos = (y_train_sarc == 1).sum()
pos_weight_value = num_neg / num_pos

print(f"\n--> Poids de la classe sarcastique (pos_weight) = {pos_weight_value:.2f}")

# 2. Définition du tenseur de poids [poids_classe_0, poids_classe_1]
weights = torch.tensor([1.0, float(pos_weight_value)], device=device)
# 3. Fonction de perte pondérée
criterion = nn.CrossEntropyLoss(weight=weights)

In [ ]:
# 2. Labels de Sentiment (texte -> entier 0, 1, 2)
sentiment_map = {"NEG": 0, "NEU": 1, "POS": 2}
y_train_sent = train_df['sentiment'].map(sentiment_map).values
y_val_sent = val_df['sentiment'].map(sentiment_map).values
y_test_sent = test_df['sentiment'].map(sentiment_map).values
print("\nDistribution Sentiment dans Train :")
print(pd.Series(y_train_sent).value_counts())

In [ ]:
from arabert.preprocess import ArabertPreprocessor

# Choix du modèle pour cette session
Arabert = "aubmindlab/bert-base-arabertv02-twitter"
Marbert = "UBC-NLP/MARBERTv2"  # Décommentez pour MARBERT


prep_Arabert = ArabertPreprocessor(model_name=Arabert)
clean_func_arabert = prep_Arabert.preprocess

prep_Marbert = ArabertPreprocessor(model_name=Marbert)
clean_func_Marbert = lambda x: str(x).strip()

In [ ]:
tokenizer_arabert = AutoTokenizer.from_pretrained("aubmindlab/bert-base-arabertv02-twitter")
tokenizer_marbert = AutoTokenizer.from_pretrained("UBC-NLP/MARBERTv2")

### Preprocess and Tokenization

In [ ]:
MAX_LEN = 128  # Longueur max d'un tweet

# 1. Nettoyage pour les deux modèles (Train, Val ET Test)
print("Nettoyage des tweets...")
for df in [train_df, val_df]:
    df["clean_tweets_arabert"] = df["tweet"].apply(clean_func_arabert)
    df["clean_tweets_marbert"] = df["tweet"].apply(clean_func_Marbert)

print("Nettoyage terminé !")


# 2. Tokenisation pour AraBERT
print("\n--- Tokenisation AraBERT ---")
train_encodings_arabert = tokenizer_arabert(
    train_df["clean_tweets_arabert"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

val_encodings_arabert = tokenizer_arabert(
    val_df["clean_tweets_arabert"].tolist(),  #  val_df ici !
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)


# 3. Tokenisation pour MARBERT
print("--- Tokenisation MARBERT ---")
train_encodings_marbert = tokenizer_marbert(
    train_df["clean_tweets_marbert"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

val_encodings_marbert = tokenizer_marbert(
    val_df["clean_tweets_marbert"].tolist(),  #  val_df ici !
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

print("\n Toutes les tokenisations sont prêtes et ont les bonnes dimensions !")


In [ ]:
train_sarc_tensor = torch.tensor(y_train_sarc, dtype=torch.long)
val_sarc_tensor   = torch.tensor(y_val_sarc, dtype=torch.long)

train_sent_tensor = torch.tensor(y_train_sent, dtype=torch.long)
val_sent_tensor   = torch.tensor(y_val_sent, dtype=torch.long)


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 32

# ==========================================
# 1. DATALOADERS POUR ARABERT
# ==========================================
train_dataset_arabert = TensorDataset(
    train_encodings_arabert['input_ids'], 
    train_encodings_arabert['attention_mask'], 
    train_sarc_tensor, 
    train_sent_tensor
)
val_dataset_arabert = TensorDataset(
    val_encodings_arabert['input_ids'], 
    val_encodings_arabert['attention_mask'], 
    val_sarc_tensor, 
    val_sent_tensor
)

train_loader_arabert = DataLoader(train_dataset_arabert, batch_size=BATCH_SIZE, shuffle=True)
val_loader_arabert   = DataLoader(val_dataset_arabert, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders créés avec succès ! (Batches Train : {len(train_loader_arabert)})")


In [ ]:
# ==========================================
# 2. DATALOADERS POUR MARBERT
# ==========================================
train_dataset_marbert = TensorDataset(
    train_encodings_marbert['input_ids'], 
    train_encodings_marbert['attention_mask'], 
    train_sarc_tensor, 
    train_sent_tensor
)
val_dataset_marbert = TensorDataset(
    val_encodings_marbert['input_ids'], 
    val_encodings_marbert['attention_mask'], 
    val_sarc_tensor, 
    val_sent_tensor
)

train_loader_marbert = DataLoader(train_dataset_marbert, batch_size=BATCH_SIZE, shuffle=True)
val_loader_marbert   = DataLoader(val_dataset_marbert, batch_size=BATCH_SIZE, shuffle=False)


forward() donne la recette de cuisine : elle explique comment les données entrent, traversent chaque couche l'une après l'autre, et ressortent sous forme de prédiction

In [ ]:
class SarcasmClassifier(nn.Module):
    def __init__(self,model_name, dropout_rate= 0.2) :
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(h,2)
    def forward(self, input_ids, attention_mask):
       # Passage dans BERT
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        
        # Récupération du vecteur [CLS] (indice 0 de la séquence)
        cls_rep = outputs.last_hidden_state[:, 0, :]
        
        # Classification
        logits = self.classifier(self.dropout(cls_rep))
        return logits
    

### Marbert monotache

In [ ]:
from transformers import get_linear_schedule_with_warmup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHOSEN_MODEL = "UBC-NLP/MARBERTv2" 
model = SarcasmClassifier(CHOSEN_MODEL).to(DEVICE)

pos_weight = (y_train_sarc == 0).sum() / (y_train_sarc == 1).sum()
class_weights = torch.tensor([1.0, float(pos_weight)], device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)

EPOCHS = 3
LR = 2e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

current_train_loader = train_loader_marbert 
total_steps = len(current_train_loader) * EPOCHS
warmup_steps = int(0.10 * total_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))


In [ ]:
from sklearn.metrics import f1_score
import numpy as np
def evaluate_and_find_threshold(model, dataloader, device):
    model.eval()
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for batch in dataloader:
            # Récupérer input_ids, attention_mask, y_sarc
            b_ids = batch[0].to(device)
            b_mask = batch[1].to(device)
            b_sarc = batch[2].to(device)
            logits = model(b_ids, b_mask)
            # Probabilité de la classe 1 (sarcastique)
            probs = torch.softmax(logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(b_sarc.cpu().numpy())
    all_probs = np.array(all_probs)
    all_targets = np.array(all_targets)
    # Recherche du meilleur seuil
    best_thresh = 0.5
    best_f1 = -1.0
    for t in np.arange(0.10, 0.90, 0.02):
        preds = (all_probs >= t).astype(int)
        score = f1_score(all_targets, preds, pos_label=1, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = t
    return best_thresh, best_f1, all_probs, all_targets


In [ ]:
import copy

best_val_f1 = -1.0
best_threshold = 0.5
best_weights = None

current_val_loader = val_loader_marbert  # ou val_loader_arabert

print("Démarrage du Fine-Tuning...")
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    # 2. ENVELOPPER LE LOADER DANS TQDM
    pbar = tqdm(current_train_loader, desc=f"Époque {epoch}/{EPOCHS}", leave=True)
    for step, batch in enumerate(current_train_loader):
        b_ids  = batch[0].to(DEVICE)
        b_mask = batch[1].to(DEVICE)
        b_sarc = batch[2].to(DEVICE)

        optimizer.zero_grad()

        # Calcul en FP16
        with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
            logits = model(b_ids, b_mask)
            loss = criterion(logits, b_sarc)

        # Rétropropagation
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})
    avg_train_loss = total_loss / len(current_train_loader)

    # Évaluation sur la validation après chaque époque
    opt_thresh, val_f1, _, _ = evaluate_and_find_threshold(model, current_val_loader, DEVICE)
    print(f"Époque {epoch}/{EPOCHS} | Perte Train: {avg_train_loss:.4f} | Val F1-sarc: {val_f1:.4f} (Seuil optimal: {opt_thresh:.2f})")

    # Early stopping : on garde le modèle ayant le meilleur F1-sarcastique sur val
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_threshold = opt_thresh
        best_weights = copy.deepcopy(model.state_dict())

# Recharger les meilleurs poids trouvés
model.load_state_dict(best_weights)
print(f"\n--> Entraînement terminé ! Meilleur F1-sarc sur Val: {best_val_f1:.4f} avec seuil: {best_threshold:.2f}")


### Marbert Multitache

In [ ]:

class MultiTaskModel(nn.Module):
    def __init__(self, model_name, dropout_prob=0.2):
        super().__init__()
        # 1. Encodeur partagé (AraBERT ou MARBERT)
        self.enc = AutoModel.from_pretrained(model_name)
        h = self.enc.config.hidden_size  # 768
        
        self.dropout = nn.Dropout(dropout_prob)
        
        # 2. Les deux têtes linéaires spécialisées
        self.sarc = nn.Linear(h, 2)  # Sarcasme : 2 classes (0: non, 1: oui)
        self.sent = nn.Linear(h, 3)  # Sentiment : 3 classes (0: NEG, 1: NEU, 2: POS)
    def forward(self, input_ids, attention_mask):
        # Passage dans l'encodeur partagé
        out = self.enc(input_ids=input_ids, attention_mask=attention_mask)
        
        # Vecteur du token [CLS] (indice 0)
        cls_token = out.last_hidden_state[:, 0, :]
        cls_drop = self.dropout(cls_token)
        
        # Retourne les logits des deux têtes
        logits_sarc = self.sarc(cls_drop)
        logits_sent = self.sent(cls_drop)
        return logits_sarc, logits_sent

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def evaluate_multitask(model, dataloader, device):
    model.eval()
    all_sarc_probs = []
    all_sarc_true = []
    all_sent_preds = []
    all_sent_true = []

    with torch.no_grad():
        for batch in dataloader:
            b_ids  = batch[0].to(device)
            b_mask = batch[1].to(device)
            b_sarc = batch[2].to(device)
            b_sent = batch[3].to(device)

            logits_sarc, logits_sent = model(b_ids, b_mask)

            # Probabilité de la classe 1 (sarcastique)
            probs_sarc = torch.softmax(logits_sarc, dim=1)[:, 1]
            all_sarc_probs.extend(probs_sarc.cpu().numpy())
            all_sarc_true.extend(b_sarc.cpu().numpy())

            # Prédictions du sentiment (classe avec le max logit)
            preds_sent = torch.argmax(logits_sent, dim=1)
            all_sent_preds.extend(preds_sent.cpu().numpy())
            all_sent_true.extend(b_sent.cpu().numpy())

    all_sarc_probs = np.array(all_sarc_probs)
    all_sarc_true = np.array(all_sarc_true)
    all_sent_preds = np.array(all_sent_preds)
    all_sent_true = np.array(all_sent_true)

    # 1. Recherche du seuil optimal pour maximiser le F1-sarcastique
    best_thresh = 0.5
    best_f1_sarc = -1.0
    for t in np.arange(0.10, 0.90, 0.02):
        preds = (all_sarc_probs >= t).astype(int)
        score = f1_score(all_sarc_true, preds, pos_label=1, zero_division=0)
        if score > best_f1_sarc:
            best_f1_sarc = score
            best_thresh = t

    # 2. Score F1-macro pour le sentiment
    sent_f1_macro = f1_score(all_sent_true, all_sent_preds, average='macro', zero_division=0)

    return best_thresh, best_f1_sarc, sent_f1_macro, all_sarc_probs


In [ ]:
import copy
from transformers import get_linear_schedule_with_warmup

def train_multitask(
    model, 
    train_loader, 
    val_loader, 
    device, 
    lam=0.5, 
    epochs=3, 
    lr=2e-5, 
    pos_weight=None
):
    # Perte pondérée pour le sarcasme
    if pos_weight is not None:
        weights = torch.tensor([1.0, float(pos_weight)], device=device)
        ce_sarc = nn.CrossEntropyLoss(weight=weights)
    else:
        ce_sarc = nn.CrossEntropyLoss()

    # Perte standard pour le sentiment (3 classes)
    ce_sent = nn.CrossEntropyLoss()

    # Optimiseur AdamW avec weight decay 0.01
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Scheduler avec warmup 10%
    total_steps = len(train_loader) * epochs
    warmup_steps = int(0.10 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=warmup_steps, 
        num_training_steps=total_steps
    )

    # Scaler FP16
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val_f1_sarc = -1.0
    best_threshold = 0.5
    best_weights = None

    print(f"\n--- Début entraînement Multi-tâche (lambda = {lam}) ---")
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Époque {epoch}/{epochs} (λ={lam})", leave=True)
        for batch in train_loader:
            b_ids  = batch[0].to(device)
            b_mask = batch[1].to(device)
            b_sarc = batch[2].to(device)
            b_sent = batch[3].to(device)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits_sarc, logits_sent = model(b_ids, b_mask)
                
                # Perte Sarcasme + lambda * Perte Sentiment
                loss_sarc = ce_sarc(logits_sarc, b_sarc)
                loss_sent = ce_sent(logits_sent, b_sent)
                loss = loss_sarc + lam * loss_sent

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()
            # 3. AFFICHER LA PERTE EN TEMPS RÉEL
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
        avg_loss = total_loss / len(train_loader)

        # Validation à la fin de chaque époque
        thresh, val_f1_sarc, val_f1_sent, _ = evaluate_multitask(model, val_loader, device)
        print(f"Époque {epoch}/{epochs} | Loss: {avg_loss:.4f} | Val F1-sarc: {val_f1_sarc:.4f} (Seuil: {thresh:.2f}) | Val F1-sent: {val_f1_sent:.4f}")

        # Early stopping sur le F1-sarcastique
        if val_f1_sarc > best_val_f1_sarc:
            best_val_f1_sarc = val_f1_sarc
            best_threshold = thresh
            best_weights = copy.deepcopy(model.state_dict())

    # Charger le meilleur modèle
    model.load_state_dict(best_weights)
    print(f"--> Meilleur Val F1-sarc : {best_val_f1_sarc:.4f} au seuil {best_threshold:.2f}")
    return model, best_threshold, best_val_f1_sarc


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# =====================================================================
# 1. LABELS DU TEST (Sarcasme et Sentiment en tenseurs)
# =====================================================================
sentiment_map = {"NEG": 0, "NEU": 1, "POS": 2}

y_test_sarc = test_df['sarcasm'].astype(int).values
y_test_sent = test_df['sentiment'].map(sentiment_map).values

test_sarc_tensor = torch.tensor(y_test_sarc, dtype=torch.long)
test_sent_tensor = torch.tensor(y_test_sent, dtype=torch.long)


# =====================================================================
# 2. NETTOYAGE DES TWEETS DU TEST
# =====================================================================
print("Nettoyage des tweets du Test...")
test_df["clean_tweets_arabert"] = test_df["tweet"].apply(clean_func_arabert)
test_df["clean_tweets_marbert"] = test_df["tweet"].apply(clean_func_Marbert)


# =====================================================================
# 3. TOKENISATION DU TEST (Pour AraBERT et MARBERT)
# =====================================================================
MAX_LEN = 128

print("Tokenisation Test AraBERT...")
test_encodings_arabert = tokenizer_arabert(
    test_df["clean_tweets_arabert"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

print("Tokenisation Test MARBERT...")
test_encodings_marbert = tokenizer_marbert(
    test_df["clean_tweets_marbert"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)


# =====================================================================
# 4. CRÉATION DES DATALOADERS DU TEST (batch_size = 32, shuffle = False)
# =====================================================================
BATCH_SIZE = 32

# Test Loader pour AraBERT
test_dataset_arabert = TensorDataset(
    test_encodings_arabert['input_ids'], 
    test_encodings_arabert['attention_mask'], 
    test_sarc_tensor, 
    test_sent_tensor
)
test_loader_arabert = DataLoader(test_dataset_arabert, batch_size=BATCH_SIZE, shuffle=False)

# Test Loader pour MARBERT
test_dataset_marbert = TensorDataset(
    test_encodings_marbert['input_ids'], 
    test_encodings_marbert['attention_mask'], 
    test_sarc_tensor, 
    test_sent_tensor
)
test_loader_marbert = DataLoader(test_dataset_marbert, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n Tout le Test est prêt !")
print(f"- Batches Test AraBERT : {len(test_loader_arabert)}")
print(f"- Batches Test MARBERT : {len(test_loader_marbert)}")


In [ ]:
seeds = [42, 123, 999]
test_scores = []
current_test_loader = test_loader_marbert

for seed in seeds:
    print(f"\n==================== SEED {seed} ====================")
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    model = MultiTaskModel(CHOSEN_MODEL).to(DEVICE)
    
    model, best_thresh, _ = train_multitask(
        model=model,
        train_loader=current_train_loader,
        val_loader=current_val_loader,
        device=DEVICE,
        lam=best_lambda,
        epochs=3,
        lr=2e-5,
        pos_weight=pos_weight
    )
    
    # Évaluation sur le TEST avec le seuil optimal choisi sur la VAL
    _, _, _, test_sarc_probs = evaluate_multitask(model, current_test_loader, DEVICE)
    test_preds = (test_sarc_probs >= best_thresh).astype(int)
    
    f1_test = f1_score(y_test_sarc, test_preds, pos_label=1, zero_division=0)
    test_scores.append(f1_test)
    print(f"Seed {seed} -> Test F1-sarcastique = {f1_test:.4f}")

mean_test = np.mean(test_scores)
std_test = np.std(test_scores)

print("\n" + "#"*50)
print(f"SCORE FINAL MULTI-TÂCHE (3 GRAINES) SUR LE TEST :")
print(f"F1-sarcastique = {mean_test:.4f} ± {std_test:.4f}")
print("#"*50)


### all whatever was above but for arabert bruh this is too much

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

class ArabertSarcasmClassifier(nn.Module):
    def __init__(self, model_name="aubmindlab/bert-base-arabertv02-twitter", dropout_rate=0.2):
        super().__init__()
        # Encodeur AraBERT pré-entraîné
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size  # 768
        
        self.dropout = nn.Dropout(dropout_rate)
        # Tête de classification : 768 -> 2 classes (non-sarcastique: 0, sarcastique: 1)
        self.classifier = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Extraction du token [CLS] (indice 0)
        cls_token = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_token))
        return logits


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def evaluate_arabert_monotask(model, dataloader, device):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for batch in dataloader:
            b_ids  = batch[0].to(device)
            b_mask = batch[1].to(device)
            b_sarc = batch[2].to(device)

            logits = model(b_ids, b_mask)
            # Probabilité de la classe 1 (sarcastique) via softmax
            probs = torch.softmax(logits, dim=1)[:, 1]

            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(b_sarc.cpu().numpy())

    all_probs = np.array(all_probs)
    all_targets = np.array(all_targets)

    # Recherche du seuil optimal qui maximise le F1-sarcastique
    best_thresh = 0.5
    best_f1 = -1.0
    for t in np.arange(0.10, 0.90, 0.02):
        preds = (all_probs >= t).astype(int)
        score = f1_score(all_targets, preds, pos_label=1, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_thresh = t

    return best_thresh, best_f1, all_probs


In [ ]:
import copy
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

def train_arabert_monotask(
    model, 
    train_loader, 
    val_loader, 
    device, 
    epochs=3, 
    lr=2e-5, 
    pos_weight=None
):
    # Perte pondérée pour le sarcasme
    if pos_weight is not None:
        weights = torch.tensor([1.0, float(pos_weight)], device=device)
        criterion = nn.CrossEntropyLoss(weight=weights)
    else:
        criterion = nn.CrossEntropyLoss()

    # Optimiseur AdamW
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Scheduler avec 10% de warmup
    total_steps = len(train_loader) * epochs
    warmup_steps = int(0.10 * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, 
        num_warmup_steps=warmup_steps, 
        num_training_steps=total_steps
    )

    # Scaler FP16 pour accélérer sur GPU
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val_f1 = -1.0
    best_threshold = 0.5
    best_weights = None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0

        progress_bar = tqdm(
            train_loader, 
            desc=f"Époque {epoch}/{epochs}", 
            leave=True
        )
        for batch in progress_bar:
            b_ids  = batch[0].to(device)
            b_mask = batch[1].to(device)
            b_sarc = batch[2].to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(b_ids, b_mask)
                loss = criterion(logits, b_sarc)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Évaluation sur la validation à chaque époque
        thresh, val_f1, _ = evaluate_arabert_monotask(model, val_loader, device)
        print(f"Époque {epoch}/{epochs} | Perte: {avg_loss:.4f} | Val F1-sarc: {val_f1:.4f} (Seuil opt: {thresh:.2f})")

        # Early stopping : on conserve les meilleurs poids
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = thresh
            best_weights = copy.deepcopy(model.state_dict())

    # Recharger le meilleur modèle
    model.load_state_dict(best_weights)
    print(f"--> Meilleur modèle Val F1-sarc : {best_val_f1:.4f} (seuil : {best_threshold:.2f})")
    return model, best_threshold, best_val_f1


In [ ]:
import os
import random
import pandas as pd

# Créer le dossier pour les prédictions
os.makedirs("predictions", exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARABERT_MODEL = "aubmindlab/bert-base-arabertv02-twitter"

# Poids de la classe sarcastique
pos_weight_val = (y_train_sarc == 0).sum() / (y_train_sarc == 1).sum()

seeds = [42, 123, 999]
test_f1_scores = []

print("=" * 60)
print(f"LANCEMENT DU FINE-TUNING MONO-TÂCHE : AraBERT-Twitter")
print(f"Device : {DEVICE} | Poids classe positive : {pos_weight_val:.2f}")
print("=" * 60)

for seed in seeds:
    print(f"\n>>>>>>>>>>>> EXÉCUTION AVEC SEED {seed} <<<<<<<<<<<<")
    
    # 1. Fixer la graine
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # 2. Instancier le modèle
    model = ArabertSarcasmClassifier(ARABERT_MODEL).to(DEVICE)

    # 3. Entraîner le modèle
    model, best_thresh, best_val_f1 = train_arabert_monotask(
        model=model,
        train_loader=train_loader_arabert,
        val_loader=val_loader_arabert,
        device=DEVICE,
        epochs=3,
        lr=2e-5,
        pos_weight=pos_weight_val
    )

    # 4. Évaluation sur le TEST avec le seuil optimal choisi sur VAL
    _, _, test_probs = evaluate_arabert_monotask(model, test_loader_arabert, DEVICE)
    test_preds = (test_probs >= best_thresh).astype(int)

    # 5. Calcul du F1-sarcastique sur TEST
    f1_test = f1_score(y_test_sarc, test_preds, pos_label=1, zero_division=0)
    test_f1_scores.append(f1_test)
    print(f"Seed {seed} -> Score TEST F1-sarcastique (au seuil {best_thresh:.2f}) : {f1_test:.4f}")

    # 6. Sauvegarder les prédictions (exigé par le sujet du TP)
    pred_path = f"predictions/arabert_monotask_seed{seed}.csv"
    pred_df = pd.DataFrame({
        'id': test_df['id'].values if 'id' in test_df.columns else np.arange(len(test_df)),
        'y_true': y_test_sarc,
        'y_pred': test_preds,
        'probability': test_probs
    })
    pred_df.to_csv(pred_path, index=False)
    print(f"Prédictions enregistrées dans : {pred_path}")

# Résumé statistique final
mean_f1 = np.mean(test_f1_scores)
std_f1 = np.std(test_f1_scores)

print("\n" + "#" * 60)
print(f"RÉSULTAT FINAL ARABERT MONO-TÂCHE SUR LE TEST (3 GRAINES) :")
print(f"F1-sarcastique = {mean_f1:.4f} ± {std_f1:.4f}")
print("#" * 60)
